In [1]:
# 🎬 🎓 SCENARIO: “College Smart Assistant System”
# 🏫 Background Story

# A college builds an AI-powered student assistant.

# 👉 Students can ask:

# “What is my attendance?”
# “What are my marks?”

# 👉 Instead of manually checking portals,
# 👉 AI fetches it instantly.
# ================================
# STEP 1: Install Gradio
# ================================
!pip install gradio


# ================================
# STEP 2: Dummy Database (Tool Layer)
# ================================
students = {
    "101": {"name": "Rahul", "attendance": 85, "marks": 78},
    "102": {"name": "Priya", "attendance": 92, "marks": 88}
}


# ================================
# STEP 3: Tool Functions (MCP Tools)
# ================================
def get_attendance(student_id):
    if student_id in students:
        return f"📊 Attendance: {students[student_id]['attendance']}%"
    return "❌ Student not found"


def get_marks(student_id):
    if student_id in students:
        return f"📝 Marks: {students[student_id]['marks']}"
    return "❌ Student not found"


# ================================
# STEP 4: Security Layer
# ================================
def secure_access(user_id, requested_id):
    return user_id == requested_id


# ================================
# STEP 5: MCP Agent Logic
# ================================
def mcp_agent(message, student_id, history):

    # Simulate logged-in user (for demo)
    user_id = student_id

    # Security Check
    if not secure_access(user_id, student_id):
        return "🚫 Access Denied", history

    # Tool Invocation Logic
    message_lower = message.lower()

    if "attendance" in message_lower:
        response = get_attendance(student_id)

    elif "marks" in message_lower:
        response = get_marks(student_id)

    elif "hello" in message_lower or "hi" in message_lower:
        response = "👋 Hello! Ask me about attendance or marks."

    else:
        response = "🤖 I can help with attendance or marks."

    # Maintain chat history
    history.append((message, response))

    return "", history


# ================================
# STEP 6: Gradio Chat UI
# ================================
import gradio as gr

with gr.Blocks() as demo:

    gr.Markdown("# 🎓 Student MCP Agent (Gradio Version)")

    student_id = gr.Textbox(label="Enter Student ID (e.g., 101)")

    chatbot = gr.Chatbot()
    msg = gr.Textbox(label="Ask your question")

    state = gr.State([])

    msg.submit(
        mcp_agent,
        inputs=[msg, student_id, state],
        outputs=[msg, chatbot]
    )

# ================================
# STEP 7: Launch App (Public URL)
# ================================
demo.launch(share=True)

/tmp/ipykernel_2024/3783630691.py:94: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbot = gr.Chatbot()
/tmp/ipykernel_2024/3783630691.py:94: DeprecationWarning: The default value of 'allow_tags' in gr.Chatbot will be changed from False to True in Gradio 6.0. You will need to explicitly set allow_tags=False if you want to disable tags in your chatbot.
  chatbot = gr.Chatbot()


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://9012ca6c8e768661be.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [3]:
# ======================================
# STEP 1: Install Libraries
# ======================================
!pip install groq gradio


# ======================================
# STEP 2: Load API Key from Colab Secrets
# ======================================
# ======================================
# STEP 2: Load API Key using os.environ
# ======================================
import os

# 🔑 Set your API key here (only for Colab testing)
os.environ["GROQ_API_KEY"] = "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"

from groq import Groq
client = Groq(api_key=os.getenv("GROQ_API_KEY"))

# ======================================
# STEP 3: Dummy Database (Tool Layer)
# ======================================
students = {
    "101": {"name": "Rahul", "attendance": 85, "marks": 78},
    "102": {"name": "Priya", "attendance": 92, "marks": 88}
}


# ======================================
# STEP 4: Tool Functions
# ======================================
def get_attendance(student_id):
    if student_id in students:
        return f"📊 Attendance: {students[student_id]['attendance']}%"
    return "❌ Student not found"


def get_marks(student_id):
    if student_id in students:
        return f"📝 Marks: {students[student_id]['marks']}"
    return "❌ Student not found"


# ======================================
# STEP 5: MCP Tool Decision via LLM
# ======================================
def decide_tool(query):
    try:
        prompt = f"""
        You are an AI assistant.

        Decide which function to call:
        - get_attendance
        - get_marks

        Rules:
        - If user asks about attendance → get_attendance
        - If user asks about marks → get_marks

        Only return function name.

        Query: {query}
        """

        response = client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=[{"role": "user", "content": prompt}]
        )

        tool = response.choices[0].message.content.strip().lower()

        return tool

    except Exception as e:
        print("❌ Groq Error:", e)
        return "fallback"


# ======================================
# STEP 6: MCP Agent (CORE LOGIC)
# ======================================
def mcp_agent(message, student_id, history):

    # Validate input
    if not student_id:
        response = "⚠️ Please enter Student ID"
        history.append((message, response))
        return "", history

    # Step 1: LLM decides tool
    tool = decide_tool(message)

    # Step 2: Tool Invocation
    if "attendance" in tool:
        response = get_attendance(student_id)

    elif "marks" in tool:
        response = get_marks(student_id)

    # Fallback (if LLM fails)
    elif tool == "fallback":
        if "attendance" in message.lower():
            response = get_attendance(student_id)
        elif "marks" in message.lower():
            response = get_marks(student_id)
        else:
            response = "⚠️ LLM failed, and I couldn't understand."

    else:
        response = "🤖 I can help with attendance or marks."

    # Step 3: Save chat
    history.append((message, response))

    return "", history


# ======================================
# STEP 7: Gradio UI
# ======================================
import gradio as gr

with gr.Blocks() as demo:

    gr.Markdown("# 🚀 MCP Agent with Groq (Stable Version)")

    student_id = gr.Textbox(label="Enter Student ID (101 / 102)")

    chatbot = gr.Chatbot(height=400)
    msg = gr.Textbox(label="Ask your question")

    state = gr.State([])

    msg.submit(
        mcp_agent,
        inputs=[msg, student_id, state],
        outputs=[msg, chatbot]
    )


# ======================================
# STEP 8: Launch App
# ======================================
demo.launch(share=True)

/tmp/ipykernel_2024/3043769175.py:130: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbot = gr.Chatbot(height=400)
/tmp/ipykernel_2024/3043769175.py:130: DeprecationWarning: The default value of 'allow_tags' in gr.Chatbot will be changed from False to True in Gradio 6.0. You will need to explicitly set allow_tags=False if you want to disable tags in your chatbot.
  chatbot = gr.Chatbot(height=400)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://ca0d7de37f89a9f31d.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
